In [33]:
from dotenv import load_dotenv
load_dotenv()

True

In [34]:
from anthropic import Anthropic
client = Anthropic()

In [59]:
def chat(messages,stop_sequences=None):
    output = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=500,
        messages=messages,
        stop_sequences=stop_sequences

    )
    return output.content[0].text

def add_user_message(messages, text):
    messages.append({
        "role": "user",
        "content": text
    })

def add_assistant_message(messages, text):
    messages.append({
        "role": "assistant",
        "content": text
    })

In [36]:
prompt = """
Genrate evaluation dataset for prompt evaluation.
This dataset will be used to evaluate prompts created for generating simple onle lines code in either Python or SQL.
Keep dataset with only three tasks.
Do not add any commentry or examples , just tasks and nothing else.

Example Output:
```json
[
{
"task" : "Description of task in brief."
},
...additional tasks

]
```

- Keeps tasks very simple and need one line of code.
"""

messages = []

add_user_message(messages, prompt)
add_assistant_message(messages, text="```json")
result = chat(messages)
print(result)




[
{
"task": "Generate a Python one-liner to reverse a string."
},
{
"task": "Generate a SQL one-liner to find the maximum salary from an employees table."
},
{
"task": "Generate a Python one-liner to check if a number is even or odd."
}
]



In [37]:
# Write above JSON as text to json file 
import json 
with open("tasks.json", "w") as file:
    file.write(
        json.dumps(json.loads(result), indent=2)
    )

In [39]:
messages.clear()

In [40]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [61]:
def grade_by_model(test_case, output):

    # as this grader is AI model based ,we'll give prompt ourselves o the model for how to asses test cases.

    prompt = f"""
    
    You are an expert code reviewer. Your task is to evaluate the following AI-generated solution.

    Original Task:
    <task>
    {test_case["task"]}
    </task>

    Solution to Evaluate:
    <solution>
    {output}
    </solution>

    Output Format
    Provide your evaluation as a structured JSON object with the following fields, in this specific order:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement
    - "reasoning": A concise explanation of your overall assessment
    - "score": A number between 1-10

    Respond with JSON. Keep your response concise and direct.
    Example response shape:
    {{
        "strengths": string[],
        "weaknesses": string[],
        "reasoning": string,
        "score": number
    }}
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, text="```json")
    result = chat(messages, stop_sequences=["```"])
    return json.loads(result)



def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
   
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [51]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results


In [62]:
with open("tasks.json", "r") as f:
    dataset = json.load(f)

print(dataset)

results = run_eval(dataset)

[{'task': 'Generate a Python one-liner to reverse a string.'}, {'task': 'Generate a SQL one-liner to find the maximum salary from an employees table.'}, {'task': 'Generate a Python one-liner to check if a number is even or odd.'}]


In [63]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# Python One-Liners to Reverse a String\n\nHere are several options:\n\n**1. Using slicing (most Pythonic):**\n```python\ns = \"hello\"\nreversed_s = s[::-1]\nprint(reversed_s)  # \"olleh\"\n```\n\n**2. Using `reversed()` function:**\n```python\ns = \"hello\"\nreversed_s = ''.join(reversed(s))\nprint(reversed_s)  # \"olleh\"\n```\n\n**3. As a complete one-liner (input/output):**\n```python\nprint(input()[::-1])\n```\n\n**4. Using a lambda function:**\n```python\nreverse = lambda x: x[::-1]\nprint(reverse(\"hello\"))  # \"olleh\"\n```\n\n---\n\n## Recommended Solution\n**The slicing method (`s[::-1]`)** is the best choice because it's:\n- \u2705 Most readable\n- \u2705 Most efficient\n- \u2705 Most Pythonic",
    "test_case": {
      "task": "Generate a Python one-liner to reverse a string."
    },
    "score": 7,
    "reasoning": "The solution is technically sound and educational, with accurate recommendations. However, it exceeds the scope of the original task by 

In [ ]:
# Below is very good exampl of Prompt evaluation Flow 
"""
Simple Prompt Evaluation Pipeline with a Model-Based Grader
=============================================================

Each step of the eval loop is its own function:
  1. load_test_set()      -> the inputs you want to test the prompt against
  2. run_prompt()          -> runs YOUR prompt against one input
  3. grade_output()        -> a second LLM call that grades the output
  4. run_evaluation()      -> ties it all together, loops over the test set
  5. summarize_results()   -> aggregates scores into a simple report

Requires: pip install anthropic
Set your API key as an environment variable: ANTHROPIC_API_KEY
"""

import json
import statistics
from anthropic import Anthropic

client = Anthropic()  # reads ANTHROPIC_API_KEY from environment

MODEL_UNDER_TEST = "claude-sonnet-4-6"   # the model/prompt you're evaluating
GRADER_MODEL = "claude-opus-4-1"          # ideally a stronger/different model as judge


# ---------------------------------------------------------------------------
# STEP 1: Test set
# ---------------------------------------------------------------------------
def load_test_set():
    """
    Returns a list of dicts, each representing one test case.
    Swap this out for loading from a CSV/JSON file for larger test sets.
    """
    return [
        {
            "id": 1,
            "ticket_text": "My order #4521 arrived broken, the screen is cracked. "
                            "I want a replacement ASAP, this is the second time this has happened."
        },
        {
            "id": 2,
            "ticket_text": "Hi, just wondering if you ship to Canada? Thanks!"
        },
        {
            "id": 3,
            "ticket_text": "I've been charged twice for my subscription this month and "
                            "nobody has responded to my last 3 emails. This is ridiculous."
        },
    ]


# ---------------------------------------------------------------------------
# STEP 2: Run the prompt under test
# ---------------------------------------------------------------------------
def run_prompt(ticket_text):
    """
    Runs the prompt you're evaluating against a single input.
    Replace this prompt template with whatever you're actually testing.
    """
    prompt = f"""Summarize this support ticket in one sentence for a triage dashboard.

Ticket: {ticket_text}"""

    response = client.messages.create(
        model=MODEL_UNDER_TEST,
        max_tokens=200,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text.strip()


# ---------------------------------------------------------------------------
# STEP 3: Model-based grader
# ---------------------------------------------------------------------------
def grade_output(ticket_text, model_output):
    """
    Sends the (input, output) pair to a grader LLM with a rubric.
    Returns a dict of scores + reasoning, parsed from JSON.
    """
    grading_prompt = f"""You are grading a one-sentence support ticket summary for accuracy and triage-usefulness.

Original ticket: {ticket_text}
Summary to grade: {model_output}

Score on these dimensions:
1. accuracy (0-2): Does the summary avoid inventing facts not in the ticket?
2. completeness (0-2): Does it capture the key issue AND urgency/sentiment if present?
3. conciseness (0-2): Is it a single clear sentence, dashboard-appropriate?

Return ONLY valid JSON, no other text:
{{"accuracy": int, "completeness": int, "conciseness": int, "reasoning": "one sentence explaining the scores"}}"""

    response = client.messages.create(
        model=GRADER_MODEL,
        max_tokens=300,
        messages=[{"role": "user", "content": grading_prompt}],
    )
    raw = response.content[0].text.strip()

    # Strip markdown code fences if the grader wraps its JSON in them
    raw = raw.replace("```json", "").replace("```", "").strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        # Fail loudly but don't crash the whole eval run
        return {"accuracy": None, "completeness": None, "conciseness": None,
                "reasoning": f"Could not parse grader output: {raw}"}


# ---------------------------------------------------------------------------
# STEP 4: Run the full evaluation loop
# ---------------------------------------------------------------------------
def run_evaluation():
    """
    Loops over the test set: run prompt -> grade output -> collect results.
    Returns a list of result dicts, one per test case.
    """
    test_set = load_test_set()
    results = []

    for case in test_set:
        model_output = run_prompt(case["ticket_text"])
        grade = grade_output(case["ticket_text"], model_output)

        results.append({
            "id": case["id"],
            "input": case["ticket_text"],
            "output": model_output,
            "grade": grade,
        })

        print(f"[Case {case['id']}] done.")

    return results


# ---------------------------------------------------------------------------
# STEP 5: Summarize results
# ---------------------------------------------------------------------------
def summarize_results(results):
    """
    Prints a per-case breakdown and an overall average score.
    """
    dimensions = ["accuracy", "completeness", "conciseness"]
    totals = []

    print("\n=== Per-case results ===")
    for r in results:
        g = r["grade"]
        scores = [g.get(d) for d in dimensions]
        if None in scores:
            print(f"Case {r['id']}: grading failed -> {g.get('reasoning')}")
            continue

        case_total = sum(scores)
        totals.append(case_total)

        print(f"\nCase {r['id']}:")
        print(f"  Input:  {r['input']}")
        print(f"  Output: {r['output']}")
        print(f"  Scores: accuracy={g['accuracy']} completeness={g['completeness']} conciseness={g['conciseness']} "
              f"(total {case_total}/6)")
        print(f"  Reasoning: {g['reasoning']}")

    if totals:
        print("\n=== Summary ===")
        print(f"Cases graded: {len(totals)}")
        print(f"Average total score: {statistics.mean(totals):.1f} / 6")
    else:
        print("\nNo valid graded results to summarize.")


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    results = run_evaluation()
    summarize_results(results)